In [196]:
import pandas as pd

In [197]:
df = pd.read_csv("IMDB Dataset.csv")

In [198]:
df.shape

(50000, 2)

In [199]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [200]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [201]:
df.drop_duplicates(inplace=True)

In [202]:
df.shape

(49582, 2)

### Pre-processing

### 1.Convert to lower case

In [203]:
df["review"] = df["review"].str.lower()

### 2.Remove urls

In [204]:
import re

In [205]:
def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 2.Remove punctuations

In [206]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"] = df["review"].apply(remove_punctuation)

### 2.Remove HTML

In [207]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_html)

### 2.Remove stopwords

In [208]:
import nltk


nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rajneelchavan1601/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rajneelchavan1601/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rajneelchavan1601/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [209]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [210]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [211]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### Stemming

In [212]:
from nltk.stem import PorterStemmer

In [213]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [214]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


#### Encoding

In [215]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [216]:
 y = df["sentiment"]

### Vectorization

In [217]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [218]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

### Datasets and DataLoader

In [219]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [220]:
X_train.shape

(39665, 5000)

In [221]:
X_test.shape

(9917, 5000)

In [222]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [223]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [224]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [225]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

### Build RNN

In [226]:
import torch.nn as nn
import torch.optim as optim

In [227]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [228]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [229]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.1738944798707962
epoch = 2/10 and loss = 0.24405448138713837
epoch = 3/10 and loss = 0.2181217074394226
epoch = 4/10 and loss = 0.193553626537323
epoch = 5/10 and loss = 0.2908364534378052
epoch = 6/10 and loss = 0.4240514039993286
epoch = 7/10 and loss = 0.2074018269777298
epoch = 8/10 and loss = 0.2605985403060913
epoch = 9/10 and loss = 0.18374931812286377
epoch = 10/10 and loss = 0.23685826361179352


In [230]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.52989815468388
